## 8. RAG Uygulaması (Retrieval-Augmented Generation)
* Amaç: LangChain ile RAG mimarisini öğretmek.
* Senaryo: Özgün bir metin koleksiyonundan bilgi çekerek modelle cevap oluşturulması.*
* Kazandırılan: Vektör tabanlı arama, Retriever, Embedding kavramı.
* Kullanılan Bileşenler: FAISS, Chroma, RetrievalQA

In [ ]:
# ============================================================
# 1) KURULUM
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" \
               "langchain-text-splitters>=0.2" \
               transformers accelerate sentencepiece \
               pypdf faiss-cpu sentence-transformers

In [ ]:


# ============================================================
# 2) İÇE AKTARIMLAR
# ============================================================
import os, textwrap, glob
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# ============================================================
# 3) MODEL ve EMBEDDING
# ============================================================
# -- LLM (küçük ve hızlı): Qwen 0.5B Instruct
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # alternatif: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.05,
)
llm = HuggingFacePipeline(pipeline=gen_pipe)

# -- Embedding (hız/kalite dengesi iyi)
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

# ============================================================
# 4) DOKÜMANLARI YÜKLEME
#    /kaggle/input/altinda-belge-klasoru/ içindeki .txt, .md, .pdf dosyaları oku
# ============================================================
DOC_DIR = "/kaggle/input/rag-belgeleri"   # <-- kendi klasör yolunuzu girin

# DirectoryLoader PDF ve düz metinleri otomatik tespit etmez; pattern ile iki kez çağırıyoruz:
docs = []
if os.path.isdir(DOC_DIR):
    # TXT & MD
    if glob.glob(os.path.join(DOC_DIR, "**/*.txt"), recursive=True) or \
       glob.glob(os.path.join(DOC_DIR, "**/*.md"), recursive=True):
        loader_txt = DirectoryLoader(DOC_DIR, glob="**/*.txt", loader_cls=TextLoader, recursive=True, show_progress=True)
        loader_md  = DirectoryLoader(DOC_DIR, glob="**/*.md",  loader_cls=TextLoader, recursive=True, show_progress=True)
        docs += loader_txt.load() + loader_md.load()
    # PDF
    for pdf_path in glob.glob(os.path.join(DOC_DIR, "**/*.pdf"), recursive=True):
        try:
            docs += PyPDFLoader(pdf_path).load()
        except Exception as e:
            print(f"PDF okunamadı: {pdf_path} | Hata: {e}")
else:
    raise FileNotFoundError(f"Klasör bulunamadı: {DOC_DIR}")

print(f"✅ Yüklenen doküman sayısı: {len(docs)}")

# ============================================================
# 5) PARÇALAMA (Chunking)
# ============================================================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # içerik yapısına göre 500-1200 arası önerilir
    chunk_overlap=120,   # bağlam taşımak için küçük örtüşme
    separators=["\n\n", "\n", " ", ""]
)
chunks = splitter.split_documents(docs)
print(f"✅ Üretilen parça sayısı: {len(chunks)}")

# ============================================================
# 6) VEKTÖR VERİTABANI (FAISS) — OLUŞTUR & KAYDET
# ============================================================
INDEX_DIR = "/kaggle/working/faiss_index"

vectordb = FAISS.from_documents(chunks, embeddings)
os.makedirs(INDEX_DIR, exist_ok=True)
vectordb.save_local(INDEX_DIR)
print(f"✅ FAISS index kaydedildi: {INDEX_DIR}")

# (İsteyenler için: Sonradan yükleme)
# vectordb = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)

retriever = vectordb.as_retriever(search_kwargs={"k": 4})

# ============================================================
# 7) PROMPT ve RAG ZİNCİRİ
# ============================================================
prompt = PromptTemplate.from_template(textwrap.dedent("""
    Aşağıda bir kullanıcı sorusu ve belgeden getirilen ilgili parçalar (context) veriliyor.
    Sadece bu bağlamdan yararlanarak, akademik ve açık bir Türkçe ile yanıt ver.
    Bağlamda olmayan bilgi için "Metinde bu bilgi yer almıyor." de; uydurma yapma.

    Soru: {question}

    Bağlam:
    {context}

    Cevap:
"""))

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",   # kısa/orta context için yeterli
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

# ============================================================
# 8) SORU ÇALIŞTIR
# ============================================================
question = "Belgelerdeki ana problem tanımı nedir ve önerilen çözüm yaklaşımı nasıl özetlenmiştir?"
result = qa_chain({"query": question})

print("\n=== SORU ===\n", question)
print("\n=== YANIT ===\n", result["result"])

print("\n=== KAYNAK PARÇALAR ===")
for i, d in enumerate(result["source_documents"], 1):
    meta = d.metadata
    src = meta.get("source", "NA")
    page = meta.get("page", "NA")
    print(f"\n--- Parça {i} | Kaynak: {src} | Sayfa: {page} ---")
    print(textwrap.shorten(d.page_content.replace("\n", " "), width=300, placeholder="..."))
